# FUNGI-TUBE Feature Processing Workflow Summary

This notebook implements the complete data processing and feature extraction pipeline for FUNGI-TUBE growth monitoring datasets. The workflow transforms raw sensor output into a curated set of biologically meaningful, interpretable features suitable for time-series analysis, anomaly detection, or downstream machine learning.

## Workflow Overview

1. **Raw Data Import and Preprocessing**
   - Load raw FUNGI-TUBE datalog (.csv)
   - Exclude initial unstable rows
   - Compute elapsed time in hours and human-readable timestamps

2. **Signal Smoothing**
   - Apply Butterworth smoothing to gas, temperature, and capacitance sensors
   - Apply rolling mean smoothing to RGB sensor outputs

3. **Environmental Temperature Estimation**
   - Compute mean environmental temperature from multiple sensors

4. **Capacitance Normalization and Detrending**
   - Z-score normalize raw capacitance
   - Remove linear temperature dependence via regression
   - Store detrended capacitance as a proxy for biological substrate modification

5. **Relative Change Calculations**
   - Compute relative change from baseline for CO₂, VOC, RH, and substrate temperature delta

6. **RGB-Derived Feature Extraction**
   - Normalize and scale raw RGB values
   - Derive grayscale brightness, color channel ratios, HSV color space metrics, and RGB deltas

7. **Z-Score and Ratio Feature Generation**
   - Z-score normalize and offset key features for stability
   - Compute stabilized ratios using z-scored, positive-offset features (e.g., VOC/CO₂, Capacitance/VOC_RH, CO₂/Heat)
   - Calculate composite growth efficiency and metabolic balance indicators

8. **Gas Concentration and Flux Estimation**
   - Convert CO₂ and water vapor concentrations to mol/m³ using the ideal gas law
   - Estimate respiration rate (CO₂) and water vapor loss rate using diffusion-based flux modeling

9. **Kinetic Feature Extraction**
   - Apply segmented (piecewise-linear) regression to smoothed time-series features (capacitance, CO₂, RH, VOC, etc.)
   - Dynamically select number of breakpoints (constrained to biologically meaningful timescales) using AICc or BIC
   - Extract slopes, intercepts, break times, and segment durations for each fitted interval
   - Compute diagnostics (RMSE, R²) and area-under-curve (AUC) metrics for both raw and fitted curves
   - Store all kinetic metrics in a single-row accumulator dataframe for each observation

10. **Curated Feature Export**
    - Export full processed dataset with all raw, intermediate, derived, and kinetic features
    - Export curated feature set focused on biologically interpretable metrics suitable for downstream modeling

---

**Note:**
All feature definitions, equations, and biological interpretations are provided in the accompanying documentation for standardized interpretation.

## 1. Dependencies

Load the Python packages used throughout the processing workflow.

In [ ]:
# Dependencies
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.signal import butter, filtfilt
import os
import matplotlib.patches as patches
from scipy.stats import zscore
import statsmodels.api as sm
import colorsys
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
import json
from matplotlib.colors import rgb_to_hsv
from numpy.linalg import lstsq
import time

## 2. Import, Smoothing, and Temperature Features

Set `DATA_PATH` to the raw FUNGI-TUBE datalog CSV before running. This section imports the run, removes the initial unstable rows, builds elapsed time columns, smooths sensor channels, and estimates environmental temperature.

In [ ]:
# --- Parameters ---
DATA_PATH = '/Users/jacobwiniski/Desktop/Data/Run_18_datalog.csv'
EXCLUDE_INITIAL_ROWS = 5          # n initial rows to exclude

# Smoothing params
RGB_ROLLING_WINDOW = 80           # samples (for R/G/B only)
BUTTER_ORDER = 2                  # Butterworth filter order (others)
FC_CPH = 0.01                     # Butterworth cutoff [cycles/hour] (e.g., 0.10 ≈ 10h period)

# --- Data Import & Preprocessing ---
df = pd.read_csv(DATA_PATH).iloc[EXCLUDE_INITIAL_ROWS:].reset_index(drop=True)

# Time columns
start_time = df['EpochMillis'].iloc[0]
df['Hours'] = (df['EpochMillis'] - start_time) / (1000 * 60 * 60)
df['DateTime'] = pd.to_datetime(df['EpochMillis'], unit='ms')

# --- Infer sampling rate (samples per hour) ---
# (Assumes fairly regular cadence; robust to occasional jitter)
dt_min = np.median(np.diff(df['EpochMillis'])) / (1000.0 * 60.0)  # minutes per sample
if not np.isfinite(dt_min) or dt_min <= 0:
    raise ValueError("Could not infer a valid sampling interval from EpochMillis.")
fs_cph = 60.0 / dt_min  # samples per hour

# --- Helper: zero-phase Butterworth smoother (for non-RGB signals) ---
def butterworth_smooth(series, fs_cph, fc_cph=FC_CPH, order=BUTTER_ORDER):
    """
    Zero-phase low-pass Butterworth smoothing (no phase shift).
    Interpolates small gaps for stability; falls back to rolling mean if too short.
    """
    x = pd.Series(series).astype(float).copy()
    # interpolate small gaps for filtfilt stability
    x = x.interpolate(limit_direction='both')

    # Normalize cutoff to Nyquist
    nyq = fs_cph / 2.0
    wn = fc_cph / nyq
    wn = np.clip(wn, 1e-6, 0.999999)

    b, a = butter(N=order, Wn=wn, btype='low')
    # filtfilt minimum length rule of thumb
    min_len = 3 * (max(len(a), len(b)) - 1) + 1

    if len(x) <= min_len or not np.isfinite(x).all():
        # Fallback: simple centered rolling mean if the series is too short
        win = min(len(x), max(3, RGB_ROLLING_WINDOW))
        return x.rolling(window=win, center=True, min_periods=1).mean().values

    return filtfilt(b, a, x.values, method='pad')

# --- Apply Smoothing ---
smoothed_df = df.copy()
non_smoothed_cols = ['EpochMillis', 'Hours', 'DateTime']
rgb_cols = ['R', 'G', 'B']

for col in df.columns:
    if col in non_smoothed_cols:
        continue

    if col in rgb_cols:
        # Tunable rolling mean for RGB
        smoothed_df[col] = df[col].rolling(
            window=RGB_ROLLING_WINDOW, center=True, min_periods=1
        ).mean()
    else:
        # Butterworth low-pass for all other numeric signals
        try:
            smoothed_df[col] = butterworth_smooth(df[col], fs_cph)
        except Exception as e:
            # Robust fallback to rolling mean if filter fails for any reason
            win = min(len(df[col]), max(3, RGB_ROLLING_WINDOW))
            smoothed_df[col] = df[col].rolling(window=win, center=True, min_periods=1).mean()
            print(f"[Warning] Butterworth failed for '{col}' ({e}). Fell back to rolling mean.")

# --- Generate Environmental Temperature Feature ---
# Prefer °C inputs. We’ll use any available among these three:
temp_candidates = [c for c in ['IR_Ambient', 'CO2Temp', 'VOC_Temp'] if c in smoothed_df.columns]
if temp_candidates:
    # Optional plausibility filter: keep columns mostly within -20..85 °C
    valid_temp_cols = []
    for c in temp_candidates:
        v = smoothed_df[c]
        # Accept the column if ≥95% of values lie in a plausible °C range
        if (v.between(-20, 85)).mean() >= 0.95:
            valid_temp_cols.append(c)

    if valid_temp_cols:
        smoothed_df['Environmental_Temp'] = smoothed_df[valid_temp_cols].mean(axis=1)
    else:
        print("[Warning] Temperature columns present but out of plausible range; Environmental_Temp not computed.")
else:
    print("[Warning] No temperature columns found for Environmental_Temp calculation.")

# --- Unified Plotting Function with Overlaid Original & Smoothed ---
def plot_original_and_smoothed(hours, original, smoothed, col_name):
    plt.figure(figsize=(10, 4))
    plt.plot(hours, original, label=f'{col_name} (Original)', alpha=0.5)
    plt.plot(hours, smoothed, label=f'{col_name} (Smoothed)', linewidth=2)
    plt.xlabel('Hours')
    plt.ylabel(col_name)
    plt.title(f'{col_name} - Original vs. Smoothed')
    plt.legend()
    plt.grid(True)
    plt.show()

# --- Plot Original vs. Smoothed Data (all numeric columns except timekeeping) ---
for col in df.columns:
    if col in non_smoothed_cols:
        continue
    # only plot numeric-like columns
    try:
        _ = pd.to_numeric(df[col])
        plot_original_and_smoothed(df['Hours'], df[col], smoothed_df[col], col)
    except Exception:
        pass  # skip non-numeric

# --- Plot Environmental Temperature ---
if 'Environmental_Temp' in smoothed_df.columns:
    plt.figure(figsize=(10, 4))
    plt.plot(smoothed_df['Hours'], smoothed_df['Environmental_Temp'],
             label='Environmental Temp (Avg of available sensors)')
    plt.xlabel('Hours')
    plt.ylabel('Temperature (°C)')
    plt.title('Environmental Temperature Over Time')
    plt.legend()
    plt.grid(True)
    plt.show()
else:
    print("[Info] Environmental_Temp not available; skipped plotting.")

## 3. Capacitance Normalization and Detrending

Normalize raw capacitance and remove its linear relationship with ambient IR temperature so the resulting signal is easier to compare across runs.

In [ ]:
# Prepare scaled and detrended capacitance

# --- Z-Score Normalize Capacitance ---
smoothed_df['CapacitanceRaw_zscore'] = zscore(smoothed_df['CapacitanceRaw'], nan_policy='omit')

# Normalize z-score series so the first valid observation is zero
_first_valid_z = smoothed_df['CapacitanceRaw_zscore'].dropna().iloc[0]
smoothed_df['CapacitanceRaw_zscore'] = smoothed_df['CapacitanceRaw_zscore'] - _first_valid_z

# Plot Z-score Normalized Capacitance Over Time
plt.figure(figsize=(10, 4))
plt.plot(smoothed_df['Hours'], smoothed_df['CapacitanceRaw_zscore'], label='Z-score Normalized Capacitance')
plt.xlabel('Hours')
plt.ylabel('Z-score Capacitance')
plt.title('Z-score Normalized Capacitance Over Time')
plt.legend()
plt.show()

# --- Linear Detrending with Respect to IR_Ambient (NaN-safe) ---
temp_col = 'IR_Ambient'
if temp_col not in smoothed_df.columns:
    raise KeyError("IR_Ambient column not found in smoothed_df.")

# Build fit data with explicit NaN handling (prevents index misalignment)
fit_df = smoothed_df[['CapacitanceRaw_zscore', temp_col]].dropna().copy()
X_fit = sm.add_constant(fit_df[[temp_col]])
y_fit = fit_df['CapacitanceRaw_zscore']

model = sm.OLS(y_fit, X_fit).fit()

# Predict on ALL rows (aligned), residuals for detrending
X_all = sm.add_constant(smoothed_df[[temp_col]])
y_hat_all = X_all @ model.params
smoothed_df['Capacitance_detrended'] = smoothed_df['CapacitanceRaw_zscore'] - y_hat_all

# Normalize detrended capacitance so the first valid observation is zero
first_valid = smoothed_df['Capacitance_detrended'].dropna().iloc[0]
smoothed_df['Capacitance_detrended'] -= first_valid

# --- Plot Linear Relationship and Detrended Capacitance ---
fig, axs = plt.subplots(1, 2, figsize=(12, 4))

# Scatterplot with Linear Fit (IR_Ambient)
axs[0].scatter(smoothed_df[temp_col], smoothed_df['CapacitanceRaw_zscore'], alpha=0.5, label='Data')
# For a clean line, sort by temperature where both are non-NaN
plot_mask = smoothed_df[[temp_col, 'CapacitanceRaw_zscore']].notna().all(axis=1)
temp_sorted = np.sort(smoothed_df.loc[plot_mask, temp_col].values)
X_line = sm.add_constant(pd.Series(temp_sorted, name=temp_col))
axs[0].plot(temp_sorted, X_line @ model.params, color='red', linewidth=2, label='Linear Fit')
axs[0].set_xlabel('IR_Ambient (°C)')
axs[0].set_ylabel('Z-score Capacitance')
axs[0].set_title('Capacitance vs. IR_Ambient')
axs[0].legend()

# Detrended Capacitance Over Time
axs[1].plot(smoothed_df['Hours'], smoothed_df['Capacitance_detrended'], label='Detrended Capacitance', color='green')
axs[1].set_xlabel('Hours')
axs[1].set_ylabel('Detrended Capacitance (Residuals)')
axs[1].set_title('Temperature-Detrended Capacitance Over Time')
axs[1].legend()

plt.tight_layout()
plt.show()

# Print model summary
print(model.summary())

## 4. Relative-Change Features

Compute baseline-anchored relative changes for temperature, gas, VOC, and humidity features, including CO2 temperature detrending when the required columns are available.

In [ ]:
# Calculate relative change in object temp, CO2, VOC, and RH

# --- Calculate IR Temperature Delta ---
if 'IR_Object' in smoothed_df.columns and 'IR_Ambient' in smoothed_df.columns:
    smoothed_df['IR_Delta'] = smoothed_df['IR_Object'] - smoothed_df['IR_Ambient']
else:
    print("[Warning] 'IR_Object' or 'IR_Ambient' missing; IR_Delta not computed.")

# --- Relative Change from Window Baseline, Anchored to 0 at First Valid Sample ---
T0_BASELINE_HOURS = 24.0
T0_WINDOW = min(len(smoothed_df), max(3, int(T0_BASELINE_HOURS * fs_cph)))

columns_to_normalize_relative = ['CO2ppm', 'VOC_Raw', 'VOC_RH', 'IR_Delta']

relative_change_columns = []
baseline_stats = {}  # store baselines actually used per column

def window_baseline(series, w):
    """Robust baseline from the first w samples. Prefer median; fallback to mean."""
    head = series.iloc[:w].dropna()
    if len(head) == 0:
        return np.nan
    return float(np.median(head))

for col in columns_to_normalize_relative:
    if col not in smoothed_df.columns:
        print(f"[Warning] Column '{col}' not found in smoothed_df. Skipping.")
        continue

    w = min(T0_WINDOW, len(smoothed_df))
    baseline = window_baseline(smoothed_df[col], w)

    if not np.isfinite(baseline):
        print(f"[Warning] No valid data in first {w} samples for '{col}'. Skipping.")
        continue

    rel_col = f'{col}_relChange'
    # subtract baseline
    smoothed_df[rel_col] = smoothed_df[col] - baseline

    # anchor so the first *valid* value is exactly 0
    first_valid_idx = smoothed_df[rel_col].first_valid_index()
    if first_valid_idx is not None and np.isfinite(smoothed_df.at[first_valid_idx, rel_col]):
        smoothed_df[rel_col] = smoothed_df[rel_col] - smoothed_df.at[first_valid_idx, rel_col]
    else:
        print(f"[Warning] '{rel_col}' has no valid first value to anchor; left unanchored.")

    baseline_stats[col] = {"baseline": baseline, "window_used": int(w)}
    relative_change_columns.append(rel_col)
    print(f"{col} normalized to baseline {baseline:.2f} from first {w} samples and anchored at 0.")

# --- CO2 relative-change detrended strictly vs CO2Temp ---
if 'CO2ppm' in smoothed_df.columns:
    if 'CO2Temp' in smoothed_df.columns:
        # Optional plausibility check for CO2Temp (-20..85 °C typical operating range)
        temp_series = smoothed_df['CO2Temp']
        plausible = (temp_series.between(-20, 85)).mean() >= 0.90
        if not plausible:
            print("[Warning] CO2Temp values outside plausible range for >10% of samples; "
                  "skipping CO2 temp-detrended relChange.")
        else:
            fit_df = smoothed_df[['CO2ppm', 'CO2Temp']].dropna().copy()
            if len(fit_df) >= 5:
                X = sm.add_constant(fit_df[['CO2Temp']])
                y = fit_df['CO2ppm']
                co2_temp_model = sm.OLS(y, X).fit()

                # Predict across all rows (align on index)
                X_all = sm.add_constant(smoothed_df[['CO2Temp']])
                y_hat = X_all @ co2_temp_model.params

                # Residuals = temperature-detrended CO2 (using only CO2Temp)
                smoothed_df['CO2ppm_tempDetrended'] = smoothed_df['CO2ppm'] - y_hat

                # Baseline-anchored relative change on the residuals
                w = min(T0_WINDOW, len(smoothed_df))
                base_resid = window_baseline(smoothed_df['CO2ppm_tempDetrended'], w)

                if np.isfinite(base_resid):
                    out_col = 'CO2ppm_relChange_tempDetrended'
                    smoothed_df[out_col] = smoothed_df['CO2ppm_tempDetrended'] - base_resid

                    first_valid_idx = smoothed_df[out_col].first_valid_index()
                    if first_valid_idx is not None and np.isfinite(smoothed_df.at[first_valid_idx, out_col]):
                        smoothed_df[out_col] = smoothed_df[out_col] - smoothed_df.at[first_valid_idx, out_col]
                    else:
                        print(f"[Warning] '{out_col}' has no valid first value to anchor; left unanchored.")

                    baseline_stats['CO2ppm_tempDetrended'] = {
                        "baseline": float(base_resid),
                        "window_used": int(w),
                        "temp_col_used": "CO2Temp",
                        "detrend_params": {k: float(v) for k, v in co2_temp_model.params.items()}
                    }
                    relative_change_columns.append(out_col)
                    print(f"CO2ppm detrended vs CO2Temp; baseline {base_resid:.2f}; added '{out_col}'.")
                else:
                    print("[Warning] Could not compute baseline for CO2ppm_tempDetrended.")
            else:
                print("[Warning] Not enough valid samples to fit CO2~CO2Temp model (need ≥5).")
    else:
        print("[Warning] CO2Temp column not found; skipping CO2 temp-detrended relChange.")
else:
    print("[Warning] CO2ppm column not found; skipping CO2 temp-detrended relChange.")

# --- Plot Relative-Change Features ---
nrows = len(relative_change_columns)

if nrows == 0:
    print("[Info] No relative-change features available to plot. Skipping plot.")
else:
    fig, axs = plt.subplots(nrows, 1, figsize=(10, 4 * nrows), sharex=True)
    # Ensure axs is always iterable
    axs = np.atleast_1d(axs)

    for i, rel_col in enumerate(relative_change_columns):
        if rel_col not in smoothed_df.columns:
            axs[i].axis('off')
            axs[i].set_title(f"{rel_col} (not available)")
            continue
        axs[i].plot(smoothed_df['Hours'], smoothed_df[rel_col], label=rel_col)
        axs[i].set_ylabel('Relative Change (anchored)')
        axs[i].set_title(f'{rel_col} Over Time')
        axs[i].legend()
        axs[i].grid(True)

    axs[-1].set_xlabel('Hours')
    plt.tight_layout()
    plt.show()

## 5. RGB-Derived Features

Convert RGB readings into exposure-normalized color, grayscale, HSV, and channel-delta features for downstream modeling and visualization.

In [ ]:
# RGB color feature extraction (vectorized, exposure-invariant hue, robust deltas)

# --- Guards ---
for c in ['R','G','B']:
    if c not in smoothed_df.columns:
        raise KeyError(f"Missing '{c}' column required for RGB features.")

R = smoothed_df['R'].astype(float).values
G = smoothed_df['G'].astype(float).values
B = smoothed_df['B'].astype(float).values
n = len(R)

# --- Ratio-normalized RGB for exposure-invariant chroma ---
eps = 1e-12
total = np.clip(R + G + B, eps, None)
r_ratio = R / total
g_ratio = G / total
b_ratio = B / total

# HSV from ratio-normalized channels (range [0,1])
rgb_stack = np.stack([r_ratio, g_ratio, b_ratio], axis=1)  # shape (n, 3)
hsv = rgb_to_hsv(rgb_stack)                                # shape (n, 3)
Hue = hsv[:, 0]
Saturation = hsv[:, 1]
Value = hsv[:, 2]  # with ratio-norm, Value≈max(r,g,b); still exposure-invariant

# --- Grayscale (luma) on raw channels (keeps brightness dynamics) ---
Grayscale = 0.2989 * R + 0.5870 * G + 0.1140 * B

# --- Robust baseline for deltas: median of first day (or available) ---
T0_BASELINE_HOURS = 24.0
T0_WINDOW = min(len(smoothed_df), max(3, int(T0_BASELINE_HOURS * fs_cph)))
R0 = np.nanmedian(R[:T0_WINDOW])
G0 = np.nanmedian(G[:T0_WINDOW])
B0 = np.nanmedian(B[:T0_WINDOW])

Delta_R = R - R0
Delta_G = G - G0
Delta_B = B - B0

# --- Hex color for visualization only ---
# Use percentile-based scaling (robust to outliers), not series max per channel.
p99_R = np.nanpercentile(R, 99) or 1.0
p99_G = np.nanpercentile(G, 99) or 1.0
p99_B = np.nanpercentile(B, 99) or 1.0

R_vis = np.clip(R / p99_R, 0, 1)
G_vis = np.clip(G / p99_G, 0, 1)
B_vis = np.clip(B / p99_B, 0, 1)

RGB_uint8 = (np.stack([R_vis, G_vis, B_vis], axis=1) * 255).astype(np.uint8)
RGB_HexColor = ['#%02X%02X%02X' % tuple(rgb) for rgb in RGB_uint8]

# --- Write features back to frame ---
smoothed_df['RGB_HexColor'] = RGB_HexColor
smoothed_df['Grayscale'] = Grayscale
smoothed_df['R_ratio'] = r_ratio
smoothed_df['G_ratio'] = g_ratio
smoothed_df['B_ratio'] = b_ratio
smoothed_df['Hue'] = Hue
smoothed_df['Saturation'] = Saturation
smoothed_df['Value'] = Value
smoothed_df['Delta_R'] = Delta_R
smoothed_df['Delta_G'] = Delta_G
smoothed_df['Delta_B'] = Delta_B

# --- Color Evolution Plot (fast) ---
fig_color, ax_color = plt.subplots(figsize=(12, 2))
# Build a 1xN strip of RGB for imshow
color_strip = np.stack([R_vis, G_vis, B_vis], axis=1).reshape(1, n, 3)
ax_color.imshow(color_strip, aspect='auto', extent=[smoothed_df['Hours'].iloc[0],
                                                    smoothed_df['Hours'].iloc[-1], 0, 1])
ax_color.set_yticks([])
ax_color.set_xlabel('Hours')
ax_color.set_title('Color Evolution Over Time (robust-scaled RGB)')
for spine in ['top','right','left']:
    ax_color.spines[spine].set_visible(False)
plt.show()

# --- Time Series Plot of RGB-Derived Features ---
features_to_plot = ['Grayscale', 'R_ratio', 'G_ratio', 'B_ratio',
                    'Hue', 'Saturation', 'Value', 'Delta_R', 'Delta_G', 'Delta_B']

fig_features, axs = plt.subplots(len(features_to_plot), 1, figsize=(10, 3 * len(features_to_plot)), sharex=True)
for i, feature in enumerate(features_to_plot):
    axs[i].plot(smoothed_df['Hours'], smoothed_df[feature], label=feature)
    axs[i].set_ylabel(feature)
    axs[i].legend(loc='upper right')
    axs[i].grid(True)
axs[-1].set_xlabel('Hours')
fig_features.suptitle('RGB-Derived Features Over Time', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Derived Ratios and Composite Features

Create stabilized z-score, ratio, rate, respiration-efficiency, heat-adjusted-growth, and metabolic-balance features. This section also writes the offset summary CSV for transparency.

In [ ]:
# Derivative features

EPS = 1e-6  # stabilization for denominators

# --- Choose CO2 sources (prefer temp-detrended) ---
CO2_RELCHANGE_COL = 'CO2ppm_relChange_tempDetrended' if 'CO2ppm_relChange_tempDetrended' in smoothed_df.columns else 'CO2ppm_relChange'
CO2_ABS_COL       = 'CO2ppm_tempDetrended'            if 'CO2ppm_tempDetrended'            in smoothed_df.columns else 'CO2ppm'

if CO2_RELCHANGE_COL not in smoothed_df.columns:
    raise KeyError("Need a CO2 relChange column (CO2ppm_relChange_tempDetrended or CO2ppm_relChange).")
if CO2_ABS_COL not in smoothed_df.columns:
    raise KeyError("Need a CO2 absolute column (CO2ppm_tempDetrended or CO2ppm).")

smoothed_df['CO2_relchange_source_for_derivatives'] = CO2_RELCHANGE_COL
smoothed_df['CO2_abs_source_for_derivatives']       = CO2_ABS_COL

# --- Helper Function to Apply Positive Offset After Z-Scoring (NaN-safe, index-aligned) ---
def zscore_with_offset(series):
    """
    Z-score normalize a series and shift so all values are > 0.
    - Fills small gaps to avoid NaN-driven offsets
    - Returns a pandas Series aligned to the input index
    - Also returns the applied offset (float)
    """
    s = pd.Series(series).astype(float)
    s_filled = s.fillna(method='ffill').fillna(method='bfill')

    z = pd.Series(zscore(s_filled, nan_policy='omit'), index=s.index)
    z = z.where(np.isfinite(z), np.nan)

    zmin = np.nanmin(z.values) if np.isnan(z.values).sum() < len(z) else np.nan
    offset = (abs(zmin) + 1.0) if (np.isfinite(zmin) and zmin <= 0) else 0.0

    return z + offset, float(offset)

# --- Precompute and Store Z-Scored, Offset-Positive Columns ---
# Use chosen CO2 columns instead of fixed names
precompute_cols = [
    'VOC_Raw_relChange',
    CO2_RELCHANGE_COL,            # relChange (possibly temp-detrended)
    'CapacitanceRaw_zscore',      
    'VOC_RH',
    CO2_ABS_COL,                  # absolute ppm (possibly temp-detrended)
    'IR_Delta'
]

offsets = {}
available_cols = []

for col in precompute_cols:
    if col not in smoothed_df.columns:
        print(f"[Warning] '{col}' not found; skipping zscore_with_offset.")
        continue
    zpos, off = zscore_with_offset(smoothed_df[col])
    smoothed_df[f'{col}_zscore_pos'] = zpos
    offsets[col] = off
    available_cols.append(col)

# --- Calculate Ratio Features (with EPS stabilization) ---
def calculate_ratio(df, num_col, denom_col, output_prefix):
    """Calculate stabilized ratio and z-score normalized version, using zscore_pos variants."""
    num = df.get(f'{num_col}_zscore_pos')
    den = df.get(f'{denom_col}_zscore_pos')
    if num is None or den is None:
        print(f"[Warning] Missing zscore_pos for {num_col} or {denom_col}; skipping {output_prefix}.")
        return
    ratio = num / (den + EPS)
    ratio = ratio.replace([np.inf, -np.inf], np.nan)
    df[f'{output_prefix}_ratio'] = ratio
    df[f'{output_prefix}_ratio_zscore'] = pd.Series(
        zscore(ratio, nan_policy='omit'), index=ratio.index
    )

# VOC per CO2 Ratio  (use temp-detrended CO2 relChange if available)
calculate_ratio(smoothed_df, 'VOC_Raw_relChange', CO2_RELCHANGE_COL, 'VOC_CO2')

# Capacitance per VOC_RH Ratio 
calculate_ratio(smoothed_df, 'CapacitanceRaw_zscore', 'VOC_RH', 'Capacitance_VOC_RH')

# Respiration Efficiency (CO2 per Heat)  (use temp-detrended absolute CO2 if available)
calculate_ratio(smoothed_df, CO2_ABS_COL, 'IR_Delta', 'RespirationEfficiency')

# Heat-Adjusted Growth (Capacitance per Heat)  
calculate_ratio(smoothed_df, 'CapacitanceRaw_zscore', 'IR_Delta', 'HeatAdjustedGrowth')

# --- Metabolic Balance Index (stabilized denominator) ---
# Use the chosen absolute CO2 column (detrended if available)
required_for_mbi = [CO2_ABS_COL, 'VOC_RH', 'IR_Delta', 'CapacitanceRaw_zscore']
if all(f'{c}_zscore_pos' in smoothed_df.columns for c in required_for_mbi):
    num_mbi = (
        smoothed_df[f'{CO2_ABS_COL}_zscore_pos'] +
        smoothed_df['VOC_RH_zscore_pos'] +
        smoothed_df['IR_Delta_zscore_pos']
    )
    den_mbi = smoothed_df['CapacitanceRaw_zscore_zscore_pos']  
    smoothed_df['MetabolicBalanceIndex'] = (num_mbi) / (den_mbi + EPS)
    smoothed_df['MetabolicBalanceIndex'] = smoothed_df['MetabolicBalanceIndex'].replace([np.inf, -np.inf], np.nan)
else:
    print("[Warning] Missing inputs for MetabolicBalanceIndex; not computed.")

# --- Capacitance rate (d/dt of capacitance proxy, per hour) ---
# Derive dt from EpochMillis if available, else from Hours
if 'EpochMillis' in smoothed_df.columns:
    dt_s = np.gradient(smoothed_df['EpochMillis'].astype(float)) / 1000.0
else:
    dt_s = np.gradient(smoothed_df['Hours'].astype(float)) * 3600.0
dt_s = np.clip(dt_s, EPS, None)
dt_hr = dt_s / 3600.0

cap_series = smoothed_df['CapacitanceRaw_zscore'].astype(float)
# Optional light smoothing for a cleaner derivative if enough points:
try:
    from scipy.signal import savgol_filter
    if len(cap_series) >= 31:
        cap_sm = savgol_filter(cap_series.fillna(method='ffill').fillna(method='bfill'), 31, 2, mode='interp')
    else:
        cap_sm = cap_series.fillna(method='ffill').fillna(method='bfill').values
except Exception:
    cap_sm = cap_series.fillna(method='ffill').fillna(method='bfill').values

smoothed_df['Capacitance_rate'] = np.gradient(cap_sm) / dt_hr  # units: (capacitance units) per hour

# --- Plot All Features Together ---
features_to_plot = [
    ('VOC_CO2_ratio_zscore', 'Z-Score Normalized VOC per CO2 Ratio'),
    ('Capacitance_VOC_RH_ratio_zscore', 'Z-Score Normalized Capacitance/VOC_RH'),
    ('RespirationEfficiency_ratio_zscore', 'Z-Score Normalized Respiration Efficiency (CO2 per Heat)'),
    ('HeatAdjustedGrowth_ratio_zscore', 'Z-Score Normalized Heat-Adjusted Growth Indicator'),
    ('MetabolicBalanceIndex', 'Metabolic Balance Index'),
    ('Capacitance_rate', 'Capacitance Rate (per hour)'),
]

fig, axs = plt.subplots(len(features_to_plot), 1, figsize=(12, 3 * len(features_to_plot)), sharex=True)
for i, (col, title) in enumerate(features_to_plot):
    if col not in smoothed_df.columns:
        axs[i].axis('off')
        axs[i].set_title(f"{title} (not available)")
        continue
    axs[i].plot(smoothed_df['Hours'], smoothed_df[col], label=title)
    axs[i].set_ylabel('Value')
    axs[i].set_title(title)
    axs[i].legend()
    axs[i].grid(True)
axs[-1].set_xlabel('Hours')
plt.tight_layout()
plt.show()

# --- Print and Export Offsets for Transparency ---
print("\nApplied Offsets (z-score -> positive shift):")
for col in available_cols:
    print(f"  {col}: {offsets[col]:.6f}")

base_path = os.path.dirname(DATA_PATH)
original_filename = os.path.splitext(os.path.basename(DATA_PATH))[0]
offsets_path_csv = os.path.join(base_path, f"offsets_{original_filename}.csv")
pd.DataFrame(
    [{"Feature": k, "Offset": v} for k, v in offsets.items()] +
    [{"Feature": "CO2_relchange_source_for_derivatives", "Offset": np.nan},
     {"Feature": "CO2_abs_source_for_derivatives", "Offset": np.nan}]
).to_csv(offsets_path_csv, index=False)
print(f"Offsets exported to CSV:\n{offsets_path_csv}")

## 7. Respiration and Water-Loss Estimates

Estimate internal CO2, water vapor concentration, diffusion-based fluxes, respiration rates, and cumulative gas/water totals using the geometry and external-condition assumptions defined in the cell.

In [ ]:
# Estimate CO2 and H2O respiration rate
# Note: this is an estimate; geometry and external assumptions matter.

# --- Choose CO2 relative-change source (prefer temp-detrended vs CO2Temp) ---
CO2_RELCHANGE_COL = 'CO2ppm_relChange_tempDetrended' if 'CO2ppm_relChange_tempDetrended' in smoothed_df.columns else 'CO2ppm_relChange'
if CO2_RELCHANGE_COL not in smoothed_df.columns:
    raise KeyError("Neither 'CO2ppm_relChange_tempDetrended' nor 'CO2ppm_relChange' is present in smoothed_df.")

smoothed_df['CO2_relChange_source'] = CO2_RELCHANGE_COL
src_used = CO2_RELCHANGE_COL

# --- FOGM Geometry Assumptions ---
V_FOGM_m3 = 17e-6  # m³
hole_areas = [np.pi * (d / 2)**2 for d in [2.12e-3, 2.12e-3, 3.5e-3]]
hole_depths = [2e-3, 2e-3, 1.5e-3]
A_total = sum(hole_areas)
L_avg = np.average(hole_depths, weights=hole_areas)

# --- Constants ---
R = 8.314       # J/mol/K
P = 101325      # Pa
D_CO2_ref = 1.5e-5   # m²/s
D_H2O_ref = 2.5e-5   # m²/s
T_ref = 298.15       # K
M_CO2 = 44.01        # g/mol
M_H2O = 18.015       # g/mol
EPS = 1e-12

# --- External Conditions ---
C_ext_CO2_ppm = 600  # ambient assumption
if 'VOC_RH' in smoothed_df.columns:
    C_ext_RH = float(np.clip(smoothed_df['VOC_RH'].iloc[:144].mean(), 0, 100))
else:
    C_ext_RH = 50.0
    print("[Info] 'VOC_RH' not found; using 50% as external RH assumption.")

# --- Temperature ---
if 'Environmental_Temp' not in smoothed_df.columns:
    raise KeyError("Environmental_Temp is required for respiration estimation.")
T_env_C = smoothed_df['Environmental_Temp'].astype(float)
T_env_K = T_env_C + 273.15

# --- (1) Estimated Internal CO2 (ppm) ---
smoothed_df['CO2ppm_est'] = C_ext_CO2_ppm + smoothed_df[CO2_RELCHANGE_COL]

# --- Convert to mol/m³ ---
P_over_RT = P / (R * np.clip(T_env_K, 200.0, None))
smoothed_df['CO2_int_mol_m3'] = (smoothed_df['CO2ppm_est'] / 1e6) * P_over_RT
C_ext_CO2_mol_m3 = (C_ext_CO2_ppm / 1e6) * P_over_RT

# --- (2) H2O concentration ---
P_sat = 610.94 * np.exp((17.625 * T_env_C) / (T_env_C + 243.04))
RH_int = np.clip(smoothed_df['VOC_RH'].astype(float), 0, 100) if 'VOC_RH' in smoothed_df.columns else np.full(len(smoothed_df), C_ext_RH)
smoothed_df['H2O_int_mol_m3'] = (RH_int / 100.0) * (P_sat / (R * T_env_K))
C_ext_H2O_mol_m3 = (C_ext_RH / 100.0) * (P_sat / (R * T_env_K))

# --- (3) Diffusion coefficients ---
D_CO2 = D_CO2_ref * (T_env_K / T_ref) ** 1.75 * (101325.0 / P)
D_H2O = D_H2O_ref * (T_env_K / T_ref) ** 1.75 * (101325.0 / P)

# --- (4) Time step ---
if 'EpochMillis' in smoothed_df.columns:
    dt_s = np.gradient(smoothed_df['EpochMillis'].astype(float)) / 1000.0
else:
    dt_s = np.gradient(smoothed_df['Hours'].astype(float)) * 3600.0
dt_s = np.clip(dt_s, EPS, None)

# --- Derivatives ---
dC_CO2_dt = np.gradient(smoothed_df['CO2_int_mol_m3'].astype(float)) / dt_s
dC_H2O_dt = np.gradient(smoothed_df['H2O_int_mol_m3'].astype(float)) / dt_s

# --- Fluxes & Rates ---
J_CO2 = D_CO2 * (smoothed_df['CO2_int_mol_m3'] - C_ext_CO2_mol_m3) / L_avg
J_H2O = D_H2O * (smoothed_df['H2O_int_mol_m3'] - C_ext_H2O_mol_m3) / L_avg
RespirationRate_CO2 = dC_CO2_dt * V_FOGM_m3 + J_CO2 * A_total
LossRate_H2O = dC_H2O_dt * V_FOGM_m3 + J_H2O * A_total

min_rate = 0.0
RespirationRate_CO2 = np.where(np.isfinite(RespirationRate_CO2) & (RespirationRate_CO2 < min_rate), 0.0, RespirationRate_CO2)
LossRate_H2O = np.where(np.isfinite(LossRate_H2O) & (LossRate_H2O < min_rate), 0.0, LossRate_H2O)

smoothed_df['RespirationRate_CO2_mol_s'] = RespirationRate_CO2
smoothed_df['LossRate_H2O_mol_s'] = LossRate_H2O

# --- Totals ---
smoothed_df['Total_CO2_mol'] = np.cumsum(np.nan_to_num(smoothed_df['RespirationRate_CO2_mol_s']) * dt_s)
smoothed_df['Total_H2O_mol'] = np.cumsum(np.nan_to_num(smoothed_df['LossRate_H2O_mol_s']) * dt_s)
smoothed_df['Total_CO2_mg'] = smoothed_df['Total_CO2_mol'] * M_CO2 * 1000.0
smoothed_df['Total_H2O_mg'] = smoothed_df['Total_H2O_mol'] * M_H2O * 1000.0

# --- (7) Plots with distinct colors ---
# CO2 Concentration & Respiration
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(smoothed_df['Hours'], smoothed_df['CO2_int_mol_m3'],
         color='tab:blue', label='CO₂ Concentration (mol/m³)')
ax1.set_xlabel('Hours'); ax1.set_ylabel('CO₂ Concentration (mol/m³)', color='tab:blue')
ax1.tick_params(axis='y', labelcolor='tab:blue'); ax1.grid(True)

ax2 = ax1.twinx()
ax2.plot(smoothed_df['Hours'], smoothed_df['RespirationRate_CO2_mol_s'],
         color='tab:orange', label='CO₂ Respiration Rate (mol/s)')
ax2.set_ylabel('CO₂ Respiration Rate (mol/s)', color='tab:orange')
ax2.tick_params(axis='y', labelcolor='tab:orange')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.title(f'FOGM CO₂ Concentration & Respiration Rate (source: {src_used})')
fig.tight_layout(); plt.show()

# H2O Concentration & Loss
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(smoothed_df['Hours'], smoothed_df['H2O_int_mol_m3'],
         color='tab:green', label='H₂O Concentration (mol/m³)')
ax1.set_xlabel('Hours'); ax1.set_ylabel('H₂O Concentration (mol/m³)', color='tab:green')
ax1.tick_params(axis='y', labelcolor='tab:green'); ax1.grid(True)

ax2 = ax1.twinx()
ax2.plot(smoothed_df['Hours'], smoothed_df['LossRate_H2O_mol_s'],
         color='tab:red', label='H₂O Loss Rate (mol/s)')
ax2.set_ylabel('H₂O Loss Rate (mol/s)', color='tab:red')
ax2.tick_params(axis='y', labelcolor='tab:red')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.title('FOGM H₂O Concentration & Loss Rate Over Time')
fig.tight_layout(); plt.show()

# Totals
fig, ax1 = plt.subplots(figsize=(12, 5))
ax1.plot(smoothed_df['Hours'], smoothed_df['Total_CO2_mol'],
         color='tab:blue', label='CO₂ Respired (mol)')
ax1.plot(smoothed_df['Hours'], smoothed_df['Total_H2O_mol'],
         color='tab:green', label='H₂O Lost (mol)')
ax1.set_xlabel('Hours'); ax1.set_ylabel('Total (mol)')
ax1.grid(True)

ax2 = ax1.twinx()
ax2.plot(smoothed_df['Hours'], smoothed_df['Total_CO2_mg'],
         color='tab:blue', linestyle='dashed', label='CO₂ Respired (mg)')
ax2.plot(smoothed_df['Hours'], smoothed_df['Total_H2O_mg'],
         color='tab:green', linestyle='dashed', label='H₂O Lost (mg)')
ax2.set_ylabel('Total (mg)')

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc='upper left')

plt.title('Cumulative CO₂ Respiration & H₂O Loss (mol & mg)')
fig.tight_layout(); plt.show()

## 8. Time-Series Feature Exports

Write the full processed feature table and the curated feature table. Output filenames are based on `DATA_PATH` and are saved beside the source datalog.

In [ ]:
# --- Export Full and Curated Versions of smoothed_df ---

# Extract folder path and original filename (without extension)
base_path = os.path.dirname(DATA_PATH)
original_filename = os.path.splitext(os.path.basename(DATA_PATH))[0]

# Full export: includes all raw, intermediate, and derivative features
full_export_path = f'{base_path}/full_features_{original_filename}.csv'
smoothed_df.to_csv(full_export_path, index=False)
print(f"Full dataset exported to:\n{full_export_path}")

# Curated export: user defines which features to keep
curated_columns = [
    'DateTime', 'Hours', 'Environmental_Temp',
    'R', 'G', 'B', 'Delta_R', 'Delta_G', 'Delta_B', 'R_ratio', 'G_ratio', 'B_ratio',
    'RGB_HexColor','Grayscale', 'Hue', 'Saturation', 'Value',
    'Capacitance_detrended', 'CapacitanceRaw_zscore',
    'CO2ppm_relChange', 'CO2ppm_relChange_tempDetrended',
    'VOC_Raw_relChange', 'VOC_RH_relChange', 'IR_Delta_relChange',
    'VOC_CO2_ratio_zscore', 'Capacitance_VOC_RH_ratio_zscore', 'RespirationEfficiency_ratio_zscore',
    'HeatAdjustedGrowth_ratio_zscore', 'MetabolicBalanceIndex', 
    'CO2_int_mol_m3', 'RespirationRate_CO2_mol_s', 'Total_CO2_mol', 'Total_CO2_mg',
    'H2O_int_mol_m3', 'LossRate_H2O_mol_s', 'Total_H2O_mol', 'Total_H2O_mg',
    'Capacitance_rate',
]


# Keep only those curated columns that actually exist in smoothed_df
curated_columns_present = [c for c in curated_columns if c in smoothed_df.columns]

# Filter and export curated dataset
curated_export_path = f'{base_path}/curated_features_{original_filename}.csv'
smoothed_df[curated_columns_present].to_csv(curated_export_path, index=False)
print(f"Curated dataset exported to:\n{curated_export_path}")

## 9. Segmented-Regression Kinetic Features

Load the curated time-series export and fit segmented regression models to extract breakpoints, slopes, durations, fit diagnostics, and full-timeframe AUC metrics.

In [ ]:
# Segmented-regression kinetics
# This cell:
#   - Loads curated features
#   - Computes segmented-regression kinetics per series (mixed break rules)
#   - Adds slopes, break times, durations, RMSE/R2, per-segment intercepts,
#     per-segment median/min/max (on original y), and FULL-TIMEFRAME AUC (of fitted curve)
#   - Updates a single-row accumulator dataframe `kinetic_features_df` (no export here)

# Configuration
SHOW_DIAGNOSTICS = True    # set True to see per-series fit plots
MIN_SEG_PTS = 144           # minimum points per segment (e.g., ~24 h at 10-min sampling)
GRID_DENSITY = 200         # higher => finer breakpoint grid (slower)

# Locate curated file produced earlier
if "curated_export_path" in locals():
    curated_path = curated_export_path
else:
    base_path = os.path.dirname(DATA_PATH)
    original_filename = os.path.splitext(os.path.basename(DATA_PATH))[0]
    curated_path = os.path.join(base_path, f"curated_features_{original_filename}.csv")

df_cur = pd.read_csv(curated_path)

# Time vector (hours since start) for numerical stability
t_raw = df_cur["Hours"].astype(float).values
t = t_raw - float(np.nanmin(t_raw))

# Targets and break rules
two_break_targets = {
    "cap_detrended": "Capacitance_detrended",
    "cap_raw_zscore": "CapacitanceRaw_zscore", 
    "ir": "IR_Delta_relChange",
    "mbi": "MetabolicBalanceIndex",
    "co2_cum": "Total_CO2_mg",
    "h2o_cum": "Total_H2O_mg",
    "grayscale": "Grayscale",
    "re": "RespirationEfficiency_ratio_zscore",
    "hagr": "HeatAdjustedGrowth_ratio_zscore",
}
one_break_targets = {
    "rh": "VOC_RH_relChange",
    "co2_rel": "CO2ppm_relChange_tempDetrended",
}

# We will compute full-interval AUC for these keys
AUC_KEYS = {"ir", "re", "hagr", "rh", "co2_rel"}

# Helpers
def _diag_metrics(y, yhat):
    rmse = float(np.sqrt(np.mean((y - yhat) ** 2)))
    ss_tot = float(np.sum((y - np.mean(y)) ** 2))
    r2 = float(1 - np.sum((y - yhat) ** 2) / (ss_tot + 1e-12)) if ss_tot > 0 else np.nan
    return rmse, r2

def _segment_stats(prefix, feats_dict, seg_idx, y_seg):
    """Add median/min/max for a segment into feats_dict."""
    if y_seg.size == 0 or not np.isfinite(y_seg).any():
        feats_dict[f"{prefix}_seg{seg_idx}_median"] = np.nan
        feats_dict[f"{prefix}_seg{seg_idx}_min"]    = np.nan
        feats_dict[f"{prefix}_seg{seg_idx}_max"]    = np.nan
    else:
        yv = y_seg[np.isfinite(y_seg)]
        feats_dict[f"{prefix}_seg{seg_idx}_median"] = float(np.median(yv))
        feats_dict[f"{prefix}_seg{seg_idx}_min"]    = float(np.min(yv))
        feats_dict[f"{prefix}_seg{seg_idx}_max"]    = float(np.max(yv))

def seg1_fit(t, y, min_seg_pts=MIN_SEG_PTS, grid_step=None):
    """
    Continuous 2-piece line:
      y = b0 + b1*t + b2*max(0, t - tb)
    Segments:
      seg1 (t < tb):   slope = b1,            intercept = b0
      seg2 (t >= tb):  slope = b1 + b2,       intercept = b0 - b2*tb
    """
    mask = np.isfinite(t) & np.isfinite(y)
    t1, y1 = t[mask], y[mask]
    n = len(t1)
    if n < 2 * min_seg_pts + 1:
        X = np.column_stack([np.ones(n), t1])
        beta, *_ = lstsq(X, y1, rcond=None)
        yhat = X @ beta
        return {"ok": 0, "tb": np.nan, "beta": (float(beta[0]), float(beta[1]), 0.0), "t": t1, "y": y1, "yhat": yhat}

    if grid_step is None:
        grid_step = max(1, n // GRID_DENSITY)
    idxs = np.arange(min_seg_pts, n - min_seg_pts, grid_step)

    best = None
    for i in idxs:
        tb = t1[i]
        X = np.column_stack([np.ones(n), t1, np.maximum(0.0, t1 - tb)])
        beta, *_ = lstsq(X, y1, rcond=None)
        yhat = X @ beta
        rss = float(np.sum((y1 - yhat) ** 2))
        if best is None or rss < best["rss"]:
            best = {"ok": 1, "tb": float(tb), "beta": tuple(map(float, beta)), "rss": rss, "t": t1, "y": y1, "yhat": yhat}
    return best

def seg2_fit(t, y, min_seg_pts=MIN_SEG_PTS, grid_step=None):
    """
    Continuous 3-piece line:
      y = b0 + b1*t + b2*max(0, t - tb1) + b3*max(0, t - tb2), tb1 < tb2
    Segments:
      seg1 (t < tb1):                slope = b1,                  intercept = b0
      seg2 (tb1 <= t < tb2):         slope = b1 + b2,             intercept = b0 - b2*tb1
      seg3 (t >= tb2):               slope = b1 + b2 + b3,        intercept = b0 - b2*tb1 - b3*tb2
    """
    mask = np.isfinite(t) & np.isfinite(y)
    t1, y1 = t[mask], y[mask]
    n = len(t1)
    if n < 3 * min_seg_pts + 2:
        return None

    if grid_step is None:
        grid_step = max(1, n // GRID_DENSITY)
    idxs = np.arange(min_seg_pts, n - min_seg_pts, grid_step)

    best = None
    for i in idxs:
        for j in idxs:
            if j - i < min_seg_pts:
                continue
            tb1, tb2 = t1[i], t1[j]
            X = np.column_stack([np.ones(n), t1, np.maximum(0.0, t1 - tb1), np.maximum(0.0, t1 - tb2)])
            beta, *_ = lstsq(X, y1, rcond=None)
            yhat = X @ beta
            rss = float(np.sum((y1 - yhat) ** 2))
            if best is None or rss < best["rss"]:
                best = {"ok": 1, "tb1": float(tb1), "tb2": float(tb2), "beta": tuple(map(float, beta)),
                        "rss": rss, "t": t1, "y": y1, "yhat": yhat}
    return best

def _update_df_row(d: dict):
    """Add/overwrite features into a single-row accumulator `kinetic_features_df`."""
    global kinetic_features_df
    if "kinetic_features_df" in globals():
        if len(kinetic_features_df) == 0:
            kinetic_features_df.loc[0, :] = np.nan
        for k, v in d.items():
            kinetic_features_df.loc[0, k] = v
    else:
        kinetic_features_df = pd.DataFrame([d])

# Compute features
feats = {}

# Two-break targets
for key, col in two_break_targets.items():
    if col not in df_cur.columns:
        continue
    y = df_cur[col].astype(float).values
    fit = seg2_fit(t, y, min_seg_pts=MIN_SEG_PTS, grid_step=max(1, len(t) // GRID_DENSITY))

    if fit is None:
        # Fallback to one-break if too few points
        fit1 = seg1_fit(t, y, min_seg_pts=MIN_SEG_PTS, grid_step=max(1, len(t) // GRID_DENSITY))
        b0, b1, b2 = fit1["beta"]
        rmse, r2 = _diag_metrics(fit1["y"], fit1["yhat"])
        tb = fit1["tb"]

        # durations
        if np.isfinite(tb):
            i_b = int(np.argmin(np.abs(fit1["t"] - tb)))
            dur1 = float(fit1["t"][i_b] - fit1["t"][0])
            dur2 = float(fit1["t"][-1] - fit1["t"][i_b])
        else:
            dur1 = dur2 = np.nan

        # intercepts
        intercept_seg1 = float(b0)
        intercept_seg2 = float(b0 - b2 * tb) if np.isfinite(tb) else np.nan

        # segment stats on ORIGINAL data using fitted break
        if np.isfinite(tb):
            mask1 = fit1["t"] < tb
            mask2 = fit1["t"] >= tb
            _segment_stats(key, feats, 1, fit1["y"][mask1])
            _segment_stats(key, feats, 2, fit1["y"][mask2])
            feats[f"{key}_seg3_median"] = np.nan
            feats[f"{key}_seg3_min"]    = np.nan
            feats[f"{key}_seg3_max"]    = np.nan
        else:
            _segment_stats(key, feats, 1, np.array([]))
            _segment_stats(key, feats, 2, np.array([]))
            feats[f"{key}_seg3_median"] = np.nan
            feats[f"{key}_seg3_min"]    = np.nan
            feats[f"{key}_seg3_max"]    = np.nan

        # --- AUC over full timeframe (of fitted curve), if requested for this key ---
        if key in AUC_KEYS:
            feats[f"{key}_AUC_total"] = float(np.trapz(fit1["yhat"], fit1["t"]))

        feats.update({
            f"{key}_fit_ok": int(fit1["ok"]),
            f"{key}_t_break1": float(tb),
            f"{key}_t_break2": np.nan,
            f"{key}_slope_seg1": float(b1),
            f"{key}_slope_seg2": float(b1 + b2),
            f"{key}_slope_seg3": np.nan,
            f"{key}_intercept_seg1": intercept_seg1,
            f"{key}_intercept_seg2": intercept_seg2,
            f"{key}_intercept_seg3": np.nan,
            f"{key}_seg1_duration": dur1,
            f"{key}_seg2_duration": dur2,
            f"{key}_seg3_duration": np.nan,
            f"{key}_RMSE": rmse,
            f"{key}_R2": r2,
        })

        if SHOW_DIAGNOSTICS:
            plt.figure(figsize=(8, 4))
            plt.plot(t, y, ".", alpha=0.4, label="observed")
            plt.plot(fit1["t"], fit1["yhat"], "-", label="segmented (1-break fallback)")
            if np.isfinite(tb):
                plt.axvline(tb, linestyle="--", alpha=0.6, label="break1")
            plt.xlabel("Hours since start"); plt.ylabel(col); plt.title(f"{col}: segmented regression (1-break fallback)")
            plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

    else:
        b0, b1, b2, b3 = fit["beta"]
        rmse, r2 = _diag_metrics(fit["y"], fit["yhat"])

        # durations by nearest indices
        i1 = int(np.argmin(np.abs(fit["t"] - fit["tb1"])))
        i2 = int(np.argmin(np.abs(fit["t"] - fit["tb2"])))
        dur1 = float(fit["t"][i1] - fit["t"][0])
        dur2 = float(fit["t"][i2] - fit["t"][i1])
        dur3 = float(fit["t"][-1] - fit["t"][i2])

        # intercepts per segment
        intercept_seg1 = float(b0)
        intercept_seg2 = float(b0 - b2 * fit["tb1"])
        intercept_seg3 = float(b0 - b2 * fit["tb1"] - b3 * fit["tb2"])

        # segment stats on ORIGINAL data using fitted breaks
        mask1 = fit["t"] < fit["tb1"]
        mask2 = (fit["t"] >= fit["tb1"]) & (fit["t"] < fit["tb2"])
        mask3 = fit["t"] >= fit["tb2"]
        _segment_stats(key, feats, 1, fit["y"][mask1])
        _segment_stats(key, feats, 2, fit["y"][mask2])
        _segment_stats(key, feats, 3, fit["y"][mask3])

        # --- AUC over full timeframe (of fitted curve), if requested for this key ---
        if key in AUC_KEYS:
            feats[f"{key}_AUC_total"] = float(np.trapz(fit["yhat"], fit["t"]))

        feats.update({
            f"{key}_fit_ok": 1,
            f"{key}_t_break1": float(fit["tb1"]),
            f"{key}_t_break2": float(fit["tb2"]),
            f"{key}_slope_seg1": float(b1),
            f"{key}_slope_seg2": float(b1 + b2),
            f"{key}_slope_seg3": float(b1 + b2 + b3),
            f"{key}_intercept_seg1": intercept_seg1,
            f"{key}_intercept_seg2": intercept_seg2,
            f"{key}_intercept_seg3": intercept_seg3,
            f"{key}_seg1_duration": dur1,
            f"{key}_seg2_duration": dur2,
            f"{key}_seg3_duration": dur3,
            f"{key}_RMSE": rmse,
            f"{key}_R2": r2,
        })

        if SHOW_DIAGNOSTICS:
            plt.figure(figsize=(8, 4))
            plt.plot(fit["t"], fit["y"], ".", alpha=0.4, label="observed")
            plt.plot(fit["t"], fit["yhat"], "-", label="segmented (2 breaks)")
            plt.axvline(fit["tb1"], linestyle="--", alpha=0.6, label="break1")
            plt.axvline(fit["tb2"], linestyle="--", alpha=0.6, label="break2")
            plt.xlabel("Hours since start"); plt.ylabel(col); plt.title(f"{col}: segmented regression (2 breaks)")
            plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

# One-break targets
for key, col in one_break_targets.items():
    if col not in df_cur.columns:
        continue
    y = df_cur[col].astype(float).values
    fit = seg1_fit(t, y, min_seg_pts=MIN_SEG_PTS, grid_step=max(1, len(t) // GRID_DENSITY))
    b0, b1, b2 = fit["beta"]
    rmse, r2 = _diag_metrics(fit["y"], fit["yhat"])

    tb = fit["tb"]
    intercept_seg1 = float(b0)
    intercept_seg2 = float(b0 - b2 * tb) if np.isfinite(tb) else np.nan

    # segment stats on ORIGINAL data using fitted break
    if np.isfinite(tb):
        mask1 = fit["t"] < tb
        mask2 = fit["t"] >= tb
        _segment_stats(key, feats, 1, fit["y"][mask1])
        _segment_stats(key, feats, 2, fit["y"][mask2])
    else:
        _segment_stats(key, feats, 1, np.array([]))
        _segment_stats(key, feats, 2, np.array([]))

    # --- AUC over full timeframe (of fitted curve), if requested for this key ---
    if key in AUC_KEYS:
        feats[f"{key}_AUC_total"] = float(np.trapz(fit["yhat"], fit["t"]))

    feats.update({
        f"{key}_fit_ok": int(fit["ok"]),
        f"{key}_t_break1": float(tb),
        f"{key}_slope_seg1": float(b1),
        f"{key}_slope_seg2": float(b1 + b2),
        f"{key}_intercept_seg1": intercept_seg1,
        f"{key}_intercept_seg2": intercept_seg2,
        f"{key}_RMSE": rmse,
        f"{key}_R2": r2,
    })

    if SHOW_DIAGNOSTICS:
        plt.figure(figsize=(8, 4))
        plt.plot(fit["t"], fit["y"], ".", alpha=0.4, label="observed")
        plt.plot(fit["t"], fit["yhat"], "-", label="segmented (1 break)")
        if np.isfinite(tb):
            plt.axvline(tb, linestyle="--", alpha=0.6, label="break1")
        plt.xlabel("Hours since start"); plt.ylabel(col); plt.title(f"{col}: segmented regression (1 break)")
        plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

# Prepare or update accumulator DataFrame
_update_df_row(feats)

# Optional tidy "long" display
long_view = (
    kinetic_features_df
    .iloc[[0]]
    .T.reset_index()
    .rename(columns={"index": "feature", 0: "value"})
    .sort_values("feature")
)

with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', None, 'display.max_colwidth', None):
    print(long_view.to_string(index=False))

## 10. VOC Piecewise-Linear Model Selection

Fit and select the VOC piecewise-linear model using the configured information criterion, breakpoint limits, segment duration constraints, and refinement settings.

In [ ]:
# VOC piecewise-linear model selection (3–5 breaks, fast greedy + refine)

# Configuration
SELECT_CRITERION   = "BIC"   # "AICc" or "BIC"
SHOW_DIAGNOSTICS   = True
VERBOSE            = True
PRINT_EVERY        = 1        # print every m increment
MAX_SECONDS        = 60       # safety timeout (seconds); set None to disable
MIN_SEG_HOURS      = 24.0     # minimum duration of each segment (hours)
COARSE_STEP_HOURS  = 3.0      # coarse grid step when searching new break inside a segment
REFINE_STEP_HOURS  = 1.0      # local coordinate-descent step
REFINE_PASSES      = 3        # refinement sweeps over all breaks

# Smoothing to suppress oscillations but keep shape
MEDIAN_WIN_H       = 2.0
GAUSS_SIGMA_H      = 0.75
GAUSS_TRUNC_H      = 2.0

t_start_clock = time.time()

def _aicc(n, rss, k):
    if n <= k + 1:
        return np.inf
    aic = n * np.log(max(rss/n, 1e-300)) + 2*k
    return aic + (2*k*(k+1))/(n-k-1)

def _bic(n, rss, k):
    return n * np.log(max(rss/n, 1e-300)) + k*np.log(max(n,2))

def _k_params(m):  # m = #breaks
    # beta = [b0, b1, hinge_1..hinge_m] => (2+m) coefficients; plus m break locations
    return (2 + m) + m  # = 2 + 2m

# Load data
if "df_cur" in globals():
    df = df_cur.copy()
else:
    if "curated_export_path" in locals():
        curated_path = curated_export_path
    else:
        base_path = os.path.dirname(DATA_PATH)
        original_filename = os.path.splitext(os.path.basename(DATA_PATH))[0]
        curated_path = os.path.join(base_path, f"curated_features_{original_filename}.csv")
    df = pd.read_csv(curated_path)

time_col = "Hours"
cand_voc = [
    "VOC_Raw_relChange","VOC_Raw_relchange","VOC_raw_relChange","VOC_raw_relchange",
    "VOC_Raw_rel_change","VOC_raw_rel_change"
]
voc_col = next((c for c in cand_voc if c in df.columns), None)
if voc_col is None:
    voc_col = next((c for c in df.columns if "voc" in c.lower() and "raw" in c.lower() and ("rel" in c.lower() or "change" in c.lower())), None)
if time_col not in df or voc_col is None:
    raise ValueError("Could not resolve 'Hours' and VOC_Raw_relChange column.")

t_raw = df[time_col].astype(float).values
y_raw = df[voc_col].astype(float).values
mask  = np.isfinite(t_raw) & np.isfinite(y_raw)
t_raw = t_raw[mask]; y_raw = y_raw[mask]
order = np.argsort(t_raw)
t_raw = t_raw[order]; y_raw = y_raw[order]
t0    = float(np.nanmin(t_raw))
t     = t_raw - t0

# cadence
dts = np.diff(t)
dt  = float(np.nanmedian(dts)) if len(dts) else 0.1667
if not np.isfinite(dt) or dt <= 0: dt = 0.1667

# smoothing
win = max(3, int(round(MEDIAN_WIN_H/dt))); win += (win % 2 == 0)
y_med = pd.Series(y_raw).rolling(window=win, center=True, min_periods=max(1, win//2)).median().to_numpy()

def _gauss_kernel(sig_pts, half_pts):
    r = int(round(half_pts))
    x = np.arange(-r, r+1, dtype=float)
    w = np.exp(-0.5*(x/sig_pts)**2); w /= np.sum(w)
    return w

sig_pts  = max(1.0, GAUSS_SIGMA_H/dt)
half_pts = max(3.0, GAUSS_TRUNC_H/dt)
gk       = _gauss_kernel(sig_pts, half_pts)
y_smooth = np.convolve(np.pad(y_med, (len(gk)//2, len(gk)//2), mode="edge"), gk, mode="valid")

n = len(t)
if n < 10:
    raise RuntimeError("Too few points after cleaning.")

min_pts = max(3, int(round(MIN_SEG_HOURS/dt)))

def _design_matrix(tt, breaks):
    cols = [np.ones_like(tt), tt]
    for tb in breaks:
        cols.append(np.maximum(0.0, tt - tb))
    return np.column_stack(cols)

def _fit_with_breaks(breaks):
    X = _design_matrix(t, breaks)
    beta, *_ = lstsq(X, y_smooth, rcond=None)
    yhat = X @ beta
    rss  = float(np.sum((y_smooth - yhat)**2))
    return beta, yhat, rss

def _score_model(breaks):
    beta, yhat, rss = _fit_with_breaks(breaks)
    m = len(breaks)
    k = _k_params(m)
    aicc = _aicc(n, rss, k)
    bic  = _bic(n, rss, k)
    score = aicc if SELECT_CRITERION.upper()=="AICc" else bic
    return dict(beta=beta, yhat=yhat, rss=rss, AICc=aicc, BIC=bic, score=score)

def _segments_from_breaks(breaks):
    edges = [t[0]] + list(breaks) + [t[-1]]
    return [(edges[i], edges[i+1]) for i in range(len(edges)-1)]

def _grid_in_segment(a, b, step_hours):
    # convert to indices and stride
    i0 = int(np.searchsorted(t, a + MIN_SEG_HOURS/1.0, side="left"))
    i1 = int(np.searchsorted(t, b - MIN_SEG_HOURS/1.0, side="right")) - 1
    if i1 - i0 < min_pts:
        return []
    step = max(1, int(round(step_hours/dt)))
    return list(range(i0, i1+1, step))

def _insert_best_new_break(current_breaks, step_hours):
    """Search all segments for the best single new break position (coarse grid)."""
    best_b = None
    best_score = np.inf
    test_breaks = list(current_breaks)
    segs = _segments_from_breaks(current_breaks)
    for seg_idx, (a,b) in enumerate(segs):
        cand_idx = _grid_in_segment(a, b, step_hours)
        for idx in cand_idx:
            tb = float(t[idx])
            new_breaks = sorted(test_breaks + [tb])
            res = _score_model(new_breaks)
            if res["score"] < best_score:
                best_score = res["score"]; best_b = tb
    if best_b is None:
        return None, None
    final_breaks = sorted(current_breaks + [best_b])
    return final_breaks, best_score

def _refine_breaks(breaks):
    """Coordinate descent: for each break, search locally ±REFINE_STEP_HOURS."""
    br = list(breaks)
    for _ in range(REFINE_PASSES):
        changed = False
        for i in range(len(br)):
            # neighborhood bounds keep min segment length
            left_bound  = (t[0] if i==0 else br[i-1]) + MIN_SEG_HOURS
            right_bound = (br[i+1] if i < len(br)-1 else t[-1]) - MIN_SEG_HOURS
            if right_bound - left_bound <= 0:
                continue
            # build small local grid around current break
            center = br[i]
            local_candidates = [center]
            step = REFINE_STEP_HOURS
            for sgn in (-1, +1):
                for kstep in (1,2):  # small two-step probe
                    cand = center + sgn * kstep * step
                    if left_bound <= cand <= right_bound:
                        local_candidates.append(cand)
            # score locals
            best_local = center
            best_score = np.inf
            for cand in sorted(set(local_candidates)):
                trial = br.copy(); trial[i] = cand
                trial.sort()
                res = _score_model(trial)
                if res["score"] < best_score:
                    best_score = res["score"]; best_local = cand
            if best_local != br[i]:
                br[i] = best_local
                changed = True
        if not changed:
            break
    return sorted(br)

def _elapsed_ok():
    return (MAX_SECONDS is None) or ((time.time() - t_start_clock) < MAX_SECONDS)

# Greedy build 1→5 breaks, keep results for m>=3
models = {}
current_breaks = []  # start with 0 breaks (fit baseline to seed the loop)
_ = _score_model(current_breaks)  # warm up

for m_target in range(1, 6):
    if not _elapsed_ok():
        if VERBOSE: print(f"[warn] Timeout before reaching {m_target} breaks.")
        break
    new_breaks, new_score = _insert_best_new_break(current_breaks, COARSE_STEP_HOURS)
    if new_breaks is None:
        if VERBOSE: print(f"[info] Could not insert break {m_target} (constraints too tight).")
        break
    # local refinement
    new_breaks = _refine_breaks(new_breaks)
    res = _score_model(new_breaks)
    current_breaks = new_breaks

    if VERBOSE and (m_target % PRINT_EVERY == 0):
        print(f"[progress] m={m_target} breaks | {SELECT_CRITERION}={res['score']:.3f} | breaks(h)={np.round(current_breaks,2)}")

    if m_target >= 3:
        models[m_target] = res | {"breaks": current_breaks.copy(), "model": f"seg{m_target}"}

if not models:
    raise RuntimeError("No models for m in 3..5 were produced. Try lowering MIN_SEG_HOURS or COARSE_STEP_HOURS, or raise MAX_SECONDS.")

# Select best model among 3..5 breaks
best = min(models.values(), key=lambda d: d["score"])
beta   = np.asarray(best["beta"], float)
yhat   = best["yhat"]
breaks = list(best["breaks"])
model  = best["model"]
m      = len(breaks)

# per-segment slopes & intercepts
seg_slopes = []
cum = 0.0
for k in range(1, m+2):
    if k == 1:
        seg_slopes.append(float(beta[1])); cum = float(beta[1])
    else:
        cum += float(beta[k]); seg_slopes.append(float(cum))

seg_ints = [float(beta[0])]
for k in range(2, m+2):
    adj = 0.0
    for j in range(2, k+1):
        adj += float(beta[j]) * float(breaks[j-2])
    seg_ints.append(float(beta[0] - adj))

# diagnostics
rmse = float(np.sqrt(np.mean((y_smooth - yhat)**2)))
ss_tot = float(np.sum((y_smooth - np.mean(y_smooth))**2))
r2 = float(1 - np.sum((y_smooth - yhat)**2)/(ss_tot + 1e-12)) if ss_tot > 0 else np.nan
auc_total = float(np.trapz(yhat, t))

# durations
edges = [t[0]] + breaks + [t[-1]]
durations = [float(edges[i+1]-edges[i]) for i in range(len(edges)-1)]

# Update accumulator
feats = {
    "voc_model": model,
    "voc_n_breaks": int(m),
    "voc_RMSE": rmse,
    "voc_R2": r2,
    "voc_AICc": float(best["AICc"]),
    "voc_BIC": float(best["BIC"]),
    "voc_AUC_total": auc_total,
    "voc_dt_median_hours": float(dt),
    "voc_median_win_h": float(MEDIAN_WIN_H),
    "voc_gauss_sigma_h": float(GAUSS_SIGMA_H),
    "voc_min_seg_hours": float(MIN_SEG_HOURS),
    "voc_coarse_step_h": float(COARSE_STEP_HOURS),
    "voc_refine_step_h": float(REFINE_STEP_HOURS),
    "voc_refine_passes": int(REFINE_PASSES),
}
for i, tb_i in enumerate(breaks, start=1):
    feats[f"voc_break{i}_h"] = float(tb_i)
for i, s in enumerate(seg_slopes, start=1):
    feats[f"voc_slope_seg{i}"] = float(s)
for i, bint in enumerate(seg_ints, start=1):
    feats[f"voc_intercept_seg{i}"] = float(bint)
for i, d in enumerate(durations, start=1):
    feats[f"voc_seg{i}_duration_h"] = float(d)

if "kinetic_features_df" in globals():
    if len(kinetic_features_df) == 0:
        kinetic_features_df.loc[0, :] = np.nan
    for k, v in feats.items():
        kinetic_features_df.loc[0, k] = v
else:
    kinetic_features_df = pd.DataFrame([feats])

# Print diagnostics and plot
if VERBOSE:
    elapsed = time.time() - t_start_clock
    print(f"[done] Selected {model} with {m} breaks | {SELECT_CRITERION}={best['score']:.3f} | elapsed {elapsed:.1f}s")
    long_view_voc = (
        kinetic_features_df.iloc[[0]]
        .T.reset_index()
        .rename(columns={"index": "feature", 0: "value"})
        .sort_values("feature")
    )
    with pd.option_context('display.max_rows', None, 'display.max_columns', None,
                           'display.width', None, 'display.max_colwidth', None):
        print(long_view_voc[long_view_voc["feature"].str.startswith("voc_")].to_string(index=False))

if SHOW_DIAGNOSTICS:
    plt.figure(figsize=(9,4.5))
    plt.plot(t, y_raw, ".", alpha=0.35, label=f"{voc_col} (raw)")
    plt.plot(t, y_smooth, "-", alpha=0.9, label="smoothed for fit")
    plt.plot(t, yhat, "-", alpha=0.9, label=f"best {model} fit ({SELECT_CRITERION} min)")
    for tb_i in breaks:
        plt.axvline(tb_i, linestyle="--", alpha=0.8, label="break")
    plt.xlabel("Hours since start"); plt.ylabel(voc_col)
    plt.title(f"VOC piecewise-linear selection: {model} | RMSE={rmse:.3f}, R²={r2:.3f}")
    plt.legend(); plt.grid(True); plt.tight_layout(); plt.show()

## 11. Kinetic Feature Export

Write the single-row kinetic feature table beside the source datalog using the run-specific filename derived from `DATA_PATH`.

In [ ]:
# Export kinetic features

base_path = os.path.dirname(DATA_PATH)
original_filename = os.path.splitext(os.path.basename(DATA_PATH))[0]

# Build export filename
export_path = os.path.join(base_path, f"kinetic_features_{original_filename}.csv")

# Ensure we only export the single observation (row 0) as a proper DataFrame
kinetic_features_df.iloc[[0]].to_csv(export_path, index=False)

print(f"[export] Features saved to: {export_path}")